# Phase C — Feature Engineering for Next-Day PM2.5 Forecasting

**City:** Delhi

## Objective

This notebook prepares time-series features for predicting next-day (t+1)
PM2.5 concentration in Delhi.

It uses the cleaned and reindexed Delhi dataset produced in Phase A.

The notebook:
- creates the t+1 PM2.5 target
- creates lag features
- creates rolling-window features
- creates calendar features
- verifies no look-ahead leakage manually
- saves the engineered dataset for regression modelling

For a prediction made on date t, only information available on or before
date t is used to predict PM2.5 on date t+1.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
from google.colab import files

print("Please upload: delhi_daily_reindexed.csv")
uploaded = files.upload()

Please upload: delhi_daily_reindexed.csv


Saving delhi_daily_reindexed.csv to delhi_daily_reindexed.csv


In [3]:
INPUT_FILE = "delhi_daily_reindexed.csv"

df = pd.read_csv(INPUT_FILE)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())

print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset loaded successfully.
Shape: (2009, 16)
Start date: 2015-01-01 00:00:00
End date: 2020-07-01 00:00:00

Columns:
['Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'City']


,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,City
0,2015-01-01,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,41.68,14.36,24.86,9.84,472.0,Severe,Delhi
1,2015-01-02,186.18,269.55,62.09,32.87,88.14,31.83,9.54,6.65,29.97,10.55,20.09,4.29,454.0,Severe,Delhi
2,2015-01-03,87.18,131.90,25.73,30.31,47.95,69.55,10.61,2.65,19.71,3.91,10.23,1.99,143.0,Moderate,Delhi
3,2015-01-04,151.84,241.84,25.01,36.91,48.62,130.36,11.54,4.63,25.36,4.26,9.71,3.34,319.0,Very Poor,Delhi
4,2015-01-05,146.60,219.13,14.01,34.92,38.25,122.88,9.20,3.33,23.20,2.80,6.21,2.96,325.0,Very Poor,Delhi


In [4]:
assert df["Date"].is_monotonic_increasing, \
    "Dates are not in chronological order."

assert df["Date"].duplicated().sum() == 0, \
    "Duplicate dates were found."

if "City" in df.columns:
    cities = df["City"].dropna().unique()
    print("Cities found:", cities)

    assert len(cities) <= 1, \
        "More than one city found."

print("Rows:", len(df))
print("Duplicate dates:", df["Date"].duplicated().sum())

print("\nIntegrity checks passed.")

Cities found: ['Delhi']
Rows: 2009
Duplicate dates: 0

Integrity checks passed.


## 1. Create the Next-Day Target

The prediction target is PM2.5 on the following day.

For date t:

**Target = PM2.5(t+1)**

The target is never imputed.

In [5]:
df["PM2.5_tomorrow"] = df["PM2.5"].shift(-1)

df[
    ["Date", "PM2.5", "PM2.5_tomorrow"]
].head(10)

,Date,PM2.5,PM2.5_tomorrow
0,2015-01-01,313.22,186.18
1,2015-01-02,186.18,87.18
2,2015-01-03,87.18,151.84
3,2015-01-04,151.84,146.60
4,2015-01-05,146.60,149.58
5,2015-01-06,149.58,217.87
6,2015-01-07,217.87,229.90
7,2015-01-08,229.90,201.66
8,2015-01-09,201.66,221.02
9,2015-01-10,221.02,205.41


## 2. Lag Features

Historical PM2.5 observations are used as predictors.

The following features are created:

- current-day PM2.5
- 1-day lag
- 2-day lag
- 3-day lag
- 7-day lag
- 14-day lag

All these measurements occur at or before date t and are therefore available
when predicting PM2.5 on t+1.

In [6]:
df["PM2.5_current"] = df["PM2.5"]

df["PM2.5_lag1"] = df["PM2.5"].shift(1)
df["PM2.5_lag2"] = df["PM2.5"].shift(2)
df["PM2.5_lag3"] = df["PM2.5"].shift(3)
df["PM2.5_lag7"] = df["PM2.5"].shift(7)
df["PM2.5_lag14"] = df["PM2.5"].shift(14)

df[
    [
        "Date",
        "PM2.5_current",
        "PM2.5_lag1",
        "PM2.5_lag2",
        "PM2.5_lag3",
        "PM2.5_lag7",
        "PM2.5_lag14",
        "PM2.5_tomorrow"
    ]
].head(16)

,Date,PM2.5_current,PM2.5_lag1,PM2.5_lag2,PM2.5_lag3,PM2.5_lag7,PM2.5_lag14,PM2.5_tomorrow
0,2015-01-01,313.22,NaN,NaN,NaN,NaN,NaN,186.18
1,2015-01-02,186.18,313.22,NaN,NaN,NaN,NaN,87.18
2,2015-01-03,87.18,186.18,313.22,NaN,NaN,NaN,151.84
3,2015-01-04,151.84,87.18,186.18,313.22,NaN,NaN,146.60
4,2015-01-05,146.60,151.84,87.18,186.18,NaN,NaN,149.58
5,2015-01-06,149.58,146.60,151.84,87.18,NaN,NaN,217.87
6,2015-01-07,217.87,149.58,146.60,151.84,NaN,NaN,229.90
7,2015-01-08,229.90,217.87,149.58,146.60,313.22,NaN,201.66
8,2015-01-09,201.66,229.90,217.87,149.58,186.18,NaN,221.02
9,2015-01-10,221.02,201.66,229.90,217.87,87.18,NaN,205.41


## 3. Rolling Features

Rolling statistics summarize recent PM2.5 behaviour.

Because the target is PM2.5 on t+1, a rolling window ending on date t is
valid and does not contain the future target.

The following rolling features are created:

- 3-day mean
- 7-day mean
- 14-day mean
- 7-day standard deviation

In [7]:
df["PM2.5_roll3_mean"] = (
    df["PM2.5"]
    .rolling(window=3, min_periods=3)
    .mean()
)

df["PM2.5_roll7_mean"] = (
    df["PM2.5"]
    .rolling(window=7, min_periods=7)
    .mean()
)

df["PM2.5_roll14_mean"] = (
    df["PM2.5"]
    .rolling(window=14, min_periods=14)
    .mean()
)

df["PM2.5_roll7_std"] = (
    df["PM2.5"]
    .rolling(window=7, min_periods=7)
    .std()
)

df[
    [
        "Date",
        "PM2.5",
        "PM2.5_roll3_mean",
        "PM2.5_roll7_mean",
        "PM2.5_roll14_mean",
        "PM2.5_roll7_std"
    ]
].head(16)

,Date,PM2.5,PM2.5_roll3_mean,PM2.5_roll7_mean,PM2.5_roll14_mean,PM2.5_roll7_std
0,2015-01-01,313.22,NaN,NaN,NaN,NaN
1,2015-01-02,186.18,NaN,NaN,NaN,NaN
2,2015-01-03,87.18,195.526667,NaN,NaN,NaN
3,2015-01-04,151.84,141.733333,NaN,NaN,NaN
4,2015-01-05,146.60,128.540000,NaN,NaN,NaN
5,2015-01-06,149.58,149.340000,NaN,NaN,NaN
6,2015-01-07,217.87,171.350000,178.924286,NaN,71.485747
7,2015-01-08,229.90,199.116667,167.021429,NaN,48.704398
8,2015-01-09,201.66,216.476667,169.232857,NaN,50.052068
9,2015-01-10,221.02,217.526667,188.352857,NaN,37.464368


## 4. No-Look-Ahead Verification

One rolling feature is manually verified.

For a selected date t, the 3-day rolling mean must contain only:

- t-2
- t-1
- t

The PM2.5 value at t+1 is the target and must not appear in the rolling
feature.

In [8]:
check_index = 20

check_date = df.loc[check_index, "Date"]

manual_window = df.loc[
    check_index - 2 : check_index,
    ["Date", "PM2.5"]
].copy()

manual_mean = manual_window["PM2.5"].mean()

generated_mean = df.loc[
    check_index,
    "PM2.5_roll3_mean"
]

target_value = df.loc[
    check_index,
    "PM2.5_tomorrow"
]

print("Prediction date (t):", check_date.date())

print(
    "Target date (t+1):",
    (check_date + pd.Timedelta(days=1)).date()
)

print("\nPM2.5 values used:")
print(manual_window.to_string(index=False))

print("\nManual mean:", manual_mean)
print("Generated mean:", generated_mean)

match = np.isclose(
    manual_mean,
    generated_mean,
    equal_nan=True
)

print("\nValues match:", match)

print("Future target PM2.5:", target_value)

assert match, "Rolling feature verification failed."

print("\nNO-LOOK-AHEAD VERIFICATION PASSED")

Prediction date (t): 2015-01-21
Target date (t+1): 2015-01-22

PM2.5 values used:
      Date  PM2.5
2015-01-19 183.35
2015-01-20 165.63
2015-01-21 159.54

Manual mean: 169.50666666666666
Generated mean: 169.50666666666666

Values match: True
Future target PM2.5: 143.68

NO-LOOK-AHEAD VERIFICATION PASSED


## 5. Calendar Features

Calendar variables are included to represent seasonal patterns in Delhi air
pollution.

The following features are added:

- month
- day of week
- day of year

In [9]:
df["month"] = df["Date"].dt.month
df["day_of_week"] = df["Date"].dt.dayofweek
df["day_of_year"] = df["Date"].dt.dayofyear

df[
    ["Date", "month", "day_of_week", "day_of_year"]
].head()

,Date,month,day_of_week,day_of_year
0,2015-01-01,1,3,1
1,2015-01-02,1,4,2
2,2015-01-03,1,5,3
3,2015-01-04,1,6,4
4,2015-01-05,1,0,5


In [10]:
FEATURES = [
    "PM2.5_current",
    "PM2.5_lag1",
    "PM2.5_lag2",
    "PM2.5_lag3",
    "PM2.5_lag7",
    "PM2.5_lag14",
    "PM2.5_roll3_mean",
    "PM2.5_roll7_mean",
    "PM2.5_roll14_mean",
    "PM2.5_roll7_std",
    "month",
    "day_of_week",
    "day_of_year"
]

TARGET = "PM2.5_tomorrow"

print("Number of features:", len(FEATURES))

for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

Number of features: 13
- PM2.5_current
- PM2.5_lag1
- PM2.5_lag2
- PM2.5_lag3
- PM2.5_lag7
- PM2.5_lag14
- PM2.5_roll3_mean
- PM2.5_roll7_mean
- PM2.5_roll14_mean
- PM2.5_roll7_std
- month
- day_of_week
- day_of_year

Target: PM2.5_tomorrow


In [11]:
missing_summary = (
    df[FEATURES + [TARGET]]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values:")
print(missing_summary)

Missing values:
PM2.5_roll14_mean    28
PM2.5_lag14          16
PM2.5_roll7_mean     14
PM2.5_roll7_std      14
PM2.5_lag7            9
PM2.5_roll3_mean      6
PM2.5_lag3            5
PM2.5_lag2            4
PM2.5_tomorrow        3
PM2.5_lag1            3
PM2.5_current         2
month                 0
day_of_week           0
day_of_year           0
dtype: int64


In [12]:
model_df = df[
    ["Date"] + FEATURES + [TARGET]
].copy()

rows_before = len(model_df)

# Target is never imputed
model_df = (
    model_df
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

print("Rows before target filtering:", rows_before)
print("Rows after target filtering:", len(model_df))

print(
    "Rows removed:",
    rows_before - len(model_df)
)

print(
    "\nRemaining missing predictor values:",
    model_df[FEATURES].isna().sum().sum()
)

Rows before target filtering: 2009
Rows after target filtering: 2006
Rows removed: 3

Remaining missing predictor values: 96


Predictors are intentionally **not** imputed here. Median imputation is fitted on training data only, inside the modelling pipeline in Notebook 03.

In [13]:
required_columns = ["Date"] + FEATURES + [TARGET]

missing_columns = [
    col for col in required_columns
    if col not in model_df.columns
]

assert len(missing_columns) == 0
assert model_df["Date"].is_monotonic_increasing
assert model_df[TARGET].isna().sum() == 0

print("Final validation PASSED.")
print("Final shape:", model_df.shape)

model_df.head()

Final validation PASSED.
Final shape: (2006, 15)


,Date,PM2.5_current,PM2.5_lag1,PM2.5_lag2,PM2.5_lag3,PM2.5_lag7,PM2.5_lag14,PM2.5_roll3_mean,PM2.5_roll7_mean,PM2.5_roll14_mean,PM2.5_roll7_std,month,day_of_week,day_of_year,PM2.5_tomorrow
0,2015-01-01,313.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,1,186.18
1,2015-01-02,186.18,313.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,4,2,87.18
2,2015-01-03,87.18,186.18,313.22,NaN,NaN,NaN,195.526667,NaN,NaN,NaN,1,5,3,151.84
3,2015-01-04,151.84,87.18,186.18,313.22,NaN,NaN,141.733333,NaN,NaN,NaN,1,6,4,146.60
4,2015-01-05,146.60,151.84,87.18,186.18,NaN,NaN,128.540000,NaN,NaN,NaN,1,0,5,149.58


In [14]:
OUTPUT_FILE = "delhi_regression_features.csv"

model_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Shape:", model_df.shape)

Saved: delhi_regression_features.csv
Shape: (2006, 15)


In [15]:
from google.colab import files

files.download("delhi_regression_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>